In [0]:
spark

### BRONZE LAYER

In [0]:
data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),  # updated record
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),  # invalid amount
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)   # duplicate
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]
df = spark.createDataFrame(data, columns)
df.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("orders")
df.write.format("delta").mode("append").saveAsTable("orders")

In [0]:
from pyspark.sql.functions import *
df.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     102|       C002|    Mobile|Electronics|  Chennai|2024-01-02|  NULL|       2|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-03| 20000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     108|       C006|    Laptop|Electronics|Bangalore|2024-01-09|-45000|       1|
|   

### SILVER LAYER

####  data cleaning

In [0]:
df = spark.table("orders")
df=df.filter(col("amount")>0)
df = df.fillna({'amount': 0, 'city': 'Unknown'})
display(df)

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1


In [0]:
df = df.dropDuplicates()
df = df.withColumn("amount", col("amount").cast("int"))
display(df)

order_id,customer_id,product,category,city,date,amount,quantity
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1


#### latest records only with same order_id

In [0]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("order_id").orderBy(col("date").desc())
df = df.withColumn("row_num", row_number().over(window_spec))
df = df.filter(col("row_num") == 1).drop("row_num")
display(df)


order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


### GOLD LAYER

#### sales analysis

Aggregations
1. Total sales by product
2. Total sales by category
3. Total sales by city


In [0]:
df.select("product", (col("amount") * col("quantity")).alias("total")).groupBy("product").sum("total").show()

+----------+----------+
|   product|sum(total)|
+----------+----------+
|    Laptop|    105000|
|    Tablet|     22000|
|    Mobile|     48000|
|     Watch|      8000|
|Headphones|      3000|
+----------+----------+



In [0]:
df.groupBy("category").agg(sum(col("amount") * col("quantity")).alias("total_sales")).orderBy("total_sales", ascending=False).show()

+-----------+-----------+
|   category|total_sales|
+-----------+-----------+
|Electronics|     175000|
|Accessories|      11000|
+-----------+-----------+



In [0]:
df.groupBy("city").agg(sum(col("amount") * col("quantity")).alias("total_sales")).orderBy("total_sales", ascending=False).show()

+---------+-----------+
|     city|total_sales|
+---------+-----------+
|Hyderabad|      72000|
|    Delhi|      55000|
|  Chennai|      48000|
|   Mumbai|       8000|
|  Unknown|       3000|
+---------+-----------+



#### customer insights

In [0]:
df.groupBy("customer_id").agg(count("order_id")).alias("order_count").show()

+-----------+---------------+
|customer_id|count(order_id)|
+-----------+---------------+
|       C001|              2|
|       C003|              1|
|       C002|              1|
|       C004|              1|
|       C005|              1|
|       C007|              1|
+-----------+---------------+



In [0]:
df.groupBy("customer_id").agg(sum(col("amount") * col("quantity")).alias("spent")).orderBy("spent", ascending=False).show()

+-----------+-----+
|customer_id|spent|
+-----------+-----+
|       C001|72000|
|       C003|55000|
|       C007|30000|
|       C002|18000|
|       C004| 8000|
|       C005| 3000|
+-----------+-----+



#### top analysis

In [0]:
df.groupBy("product").agg(count("product").alias("product_count")).orderBy("product_count",ascending=False).show()

+----------+-------------+
|   product|product_count|
+----------+-------------+
|    Laptop|            2|
|    Mobile|            2|
|    Tablet|            1|
|     Watch|            1|
|Headphones|            1|
+----------+-------------+



In [0]:
df.groupBy("customer_id",).agg(count(col("customer_id")).alias("top_customer")).orderBy("top_customer",ascending=False).show()

+-----------+------------+
|customer_id|top_customer|
+-----------+------------+
|       C001|           2|
|       C003|           1|
|       C002|           1|
|       C004|           1|
|       C005|           1|
|       C007|           1|
+-----------+------------+

